# Brain Dance - De3DGS GPU Validation

This notebook validates the **Deformable 3D Gaussians (De3DGS)** integration on Google Colab.

**Version**: 1.1.0  
**Date**: 2026-02-09  
**Minimum Requirements**: Colab (T4 GPU), 12GB VRAM  
**Estimated Runtime**: 45-60 minutes (first run), 20-30 minutes (cached)

## What This Notebook Tests

1. **Environment Setup**: GPU detection, CUDA kernel compilation
2. **Direct Training**: De3DGS training on D-NeRF bouncingballs dataset
3. **Adapter Integration**: `Deformable3DGSAdapter` pipeline validation
4. **PLY Export**: Per-frame PLY format compliance
5. **SE(3) Validation**: 6-DoF transformation correctness
6. **User Video Test**: Upload your own video for testing
7. **Download Results**: Get PLY files, MP4 renders, and reports locally

## Sections

- **Section 0**: Configuration & Resume Detection
- **Section A**: Environment Setup
- **Section B**: Test Dataset Download
- **Section C**: Direct De3DGS Training
- **Section D**: Adapter Integration Test
- **Section E**: SE(3) 6-DoF Validation
- **Section F**: Visualization & Reporting
- **Section G**: User Video Test (Optional)
- **Section H**: Download Results
- **Section I**: Cleanup & Summary

---
## Section 0: Configuration & Resume Detection

In [ ]:
# Cell 0.1: Configuration
"""
All tunable parameters in one place.
Modify these values to customize the validation run.
"""

from dataclasses import dataclass
from typing import Optional

@dataclass
class NotebookConfig:
    """All tunable parameters in one place."""
    # Repository
    repo_url: str = "https://github.com/ujseah/brain-dance.git"
    branch: str = "feat/de3dgs-migration"
    
    # Training
    training_iterations: int = 5000  # Reduced for validation (full: 20000)
    checkpoint_interval: int = 1000
    
    # Validation thresholds
    min_psnr_synthetic: float = 30.0  # D-NeRF bouncingballs
    max_position_delta: float = 0.5   # Between consecutive frames
    max_vram_gb: float = 12.0
    
    # Paths
    checkpoint_dir: str = "/content/brain_dance_checkpoint"
    output_dir: str = "/content/brain_dance_output"
    repo_dir: str = "/content/brain-dance"

CONFIG = NotebookConfig()
print(f"Configuration loaded:")
print(f"  Repository: {CONFIG.repo_url}")
print(f"  Branch: {CONFIG.branch}")
print(f"  Training iterations: {CONFIG.training_iterations}")
print(f"  PSNR threshold: {CONFIG.min_psnr_synthetic} dB")

In [ ]:
# Cell 0.2: Resume Detection
"""
Checkpoint manager for resuming after Colab disconnects.
If you want a fresh run, call CHECKPOINT.reset()
"""

import os
import json
from pathlib import Path
from datetime import datetime

class CheckpointManager:
    """Manage notebook execution checkpoints for resume capability."""
    
    def __init__(self, checkpoint_dir: str):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.state_file = self.checkpoint_dir / "state.json"
        self.state = self._load_state()
    
    def _load_state(self) -> dict:
        if self.state_file.exists():
            return json.loads(self.state_file.read_text())
        return {"completed_sections": [], "start_time": None}
    
    def _save_state(self):
        self.state_file.write_text(json.dumps(self.state, indent=2))
    
    def is_complete(self, section: str) -> bool:
        return section in self.state["completed_sections"]
    
    def mark_complete(self, section: str):
        if section not in self.state["completed_sections"]:
            self.state["completed_sections"].append(section)
            self._save_state()
            print(f"Checkpoint saved: {section}")
    
    def reset(self):
        """Clear all checkpoints (for fresh run)."""
        self.state = {"completed_sections": [], "start_time": None}
        self._save_state()
        print("Checkpoints cleared - starting fresh")

CHECKPOINT = CheckpointManager(CONFIG.checkpoint_dir)
print(f"Checkpoint status: {len(CHECKPOINT.state['completed_sections'])} sections complete")
if CHECKPOINT.state['completed_sections']:
    print(f"  Completed: {CHECKPOINT.state['completed_sections']}")
    print("  To start fresh, run: CHECKPOINT.reset()")

---
## Section A: Environment Setup

In [ ]:
# Cell A.1: GPU Verification with Tier Detection
import torch

def get_gpu_info() -> dict:
    """Get detailed GPU information."""
    if not torch.cuda.is_available():
        raise RuntimeError(
            "NO GPU DETECTED!\n\n"
            "To fix this:\n"
            "1. Go to Runtime > Change runtime type\n"
            "2. Select 'T4 GPU' under Hardware accelerator\n"
            "3. Click Save and re-run this cell"
        )
    
    props = torch.cuda.get_device_properties(0)
    
    # Detect GPU tier for runtime estimation
    gpu_name = props.name.lower()
    if "a100" in gpu_name:
        tier = "A100"
        estimated_training_time = "8-12 min"
    elif "v100" in gpu_name:
        tier = "V100"
        estimated_training_time = "15-20 min"
    elif "t4" in gpu_name:
        tier = "T4"
        estimated_training_time = "25-35 min"
    else:
        tier = "Unknown"
        estimated_training_time = "30-45 min"
    
    return {
        "name": props.name,
        "tier": tier,
        "vram_gb": props.total_memory / 1e9,
        "cuda_version": torch.version.cuda,
        "pytorch_version": torch.__version__,
        "estimated_training_time": estimated_training_time,
    }

gpu_info = get_gpu_info()
print("=" * 50)
print("GPU VERIFICATION")
print("=" * 50)
print(f"[OK] GPU: {gpu_info['name']}")
print(f"[OK] Tier: {gpu_info['tier']}")
print(f"[OK] VRAM: {gpu_info['vram_gb']:.1f} GB")
print(f"[OK] CUDA: {gpu_info['cuda_version']}")
print(f"[OK] PyTorch: {gpu_info['pytorch_version']}")
print(f"[TIME] Estimated training time: {gpu_info['estimated_training_time']}")

if gpu_info['vram_gb'] < CONFIG.max_vram_gb:
    print(f"\n[WARN] VRAM ({gpu_info['vram_gb']:.1f} GB) is below recommended {CONFIG.max_vram_gb} GB")
    print("  Training may require reduced batch size")

CHECKPOINT.mark_complete("section_a1_gpu")

In [ ]:
# Cell A.2: Clone Repository with Retry
import subprocess
import time

def clone_with_retry(url: str, dest: str, branch: str, max_retries: int = 3):
    """Clone repository with exponential backoff retry."""
    for attempt in range(max_retries):
        try:
            if os.path.exists(dest):
                print(f"Repository already exists at {dest}")
                # Verify it's the right repo and update
                result = subprocess.run(
                    ["git", "-C", dest, "remote", "get-url", "origin"],
                    capture_output=True, text=True
                )
                if url in result.stdout:
                    print("Updating existing repository...")
                    subprocess.run(["git", "-C", dest, "fetch", "--all"], check=True)
                    subprocess.run(["git", "-C", dest, "checkout", branch], check=True)
                    subprocess.run(["git", "-C", dest, "pull", "--ff-only"], check=True)
                    subprocess.run(["git", "-C", dest, "submodule", "update", "--init", "--recursive"], check=True)
                    return True
                else:
                    print("Different repository exists, removing...")
                    subprocess.run(["rm", "-rf", dest], check=True)
            
            print(f"Cloning {url} (attempt {attempt + 1}/{max_retries})...")
            subprocess.run([
                "git", "clone", "--recursive",
                "-b", branch,
                url, dest
            ], check=True)
            return True
        
        except subprocess.CalledProcessError as e:
            wait_time = 2 ** attempt  # Exponential backoff
            print(f"Clone failed, retrying in {wait_time}s...")
            time.sleep(wait_time)
    
    raise RuntimeError(f"Failed to clone repository after {max_retries} attempts")

if CHECKPOINT.is_complete("section_a2_clone"):
    print("[SKIP] Section A.2 already complete, skipping clone...")
else:
    clone_with_retry(
        CONFIG.repo_url,
        CONFIG.repo_dir,
        CONFIG.branch
    )
    os.chdir(CONFIG.repo_dir)
    
    # Verify submodule
    if not os.path.exists("deformable3dgs/train.py"):
        raise RuntimeError("De3DGS submodule not properly initialized!")
    
    print("[OK] Repository cloned and verified")
    CHECKPOINT.mark_complete("section_a2_clone")

In [ ]:
# Cell A.3: Install Dependencies
if CHECKPOINT.is_complete("section_a3_deps"):
    print("[SKIP] Section A.3 already complete, skipping dependency install...")
else:
    print("Installing dependencies...")
    !pip install -q -r {CONFIG.repo_dir}/notebooks/requirements-colab.txt
    !pip install -q -r {CONFIG.repo_dir}/backend/requirements.txt
    print("[OK] Dependencies installed")
    CHECKPOINT.mark_complete("section_a3_deps")

In [ ]:
# Cell A.4: Compile De3DGS CUDA Kernels
if CHECKPOINT.is_complete("section_a4_cuda"):
    print("[SKIP] Section A.4 already complete, skipping CUDA compilation...")
else:
    print("Compiling De3DGS CUDA kernels...")
    print("This may take 5-10 minutes on first run.\n")
    
    # Check if setup script exists
    setup_script = f"{CONFIG.repo_dir}/scripts/setup_de3dgs.sh"
    if os.path.exists(setup_script):
        !bash {setup_script}
    else:
        # Manual compilation
        print("Setup script not found, compiling manually...")
        de3dgs_dir = f"{CONFIG.repo_dir}/deformable3dgs"
        
        # Install diff-gaussian-rasterization
        !pip install -e {de3dgs_dir}/submodules/diff-gaussian-rasterization
        
        # Install simple-knn
        !pip install -e {de3dgs_dir}/submodules/simple-knn
    
    print("[OK] CUDA kernels compiled")
    CHECKPOINT.mark_complete("section_a4_cuda")

In [ ]:
# Cell A.5: Verify Installation
print("=" * 50)
print("INSTALLATION VERIFICATION")
print("=" * 50)

errors = 0

# Check core imports
try:
    from diff_gaussian_rasterization import GaussianRasterizer
    print("[OK] diff-gaussian-rasterization")
except ImportError as e:
    print(f"[FAIL] diff-gaussian-rasterization: {e}")
    errors += 1

try:
    from simple_knn import _C
    print("[OK] simple-knn")
except ImportError as e:
    print(f"[FAIL] simple-knn: {e}")
    errors += 1

try:
    from plyfile import PlyData
    print("[OK] plyfile")
except ImportError as e:
    print(f"[FAIL] plyfile: {e}")
    errors += 1

if errors == 0:
    print("\n[OK] All installations verified!")
    CHECKPOINT.mark_complete("section_a5_verify")
else:
    raise RuntimeError(f"Installation verification failed with {errors} error(s)")

---
## Section B: Test Dataset

In [ ]:
# Cell B.1: Download D-NeRF Bouncingballs Dataset
import gdown

DATASET_URL = "https://drive.google.com/uc?id=1uHVyApwqugXTFuIRRlE4abTW8_rrVeIK"
DATASET_DIR = f"{CONFIG.repo_dir}/deformable3dgs/data/dnerf"
DATASET_ZIP = f"{DATASET_DIR}/bouncingballs.zip"
DATASET_PATH = f"{DATASET_DIR}/bouncingballs"

if CHECKPOINT.is_complete("section_b1_dataset"):
    print("[SKIP] Section B.1 already complete, skipping dataset download...")
else:
    os.makedirs(DATASET_DIR, exist_ok=True)
    
    if not os.path.exists(DATASET_PATH):
        print("Downloading D-NeRF bouncingballs dataset...")
        gdown.download(DATASET_URL, DATASET_ZIP, quiet=False)
        
        print("\nExtracting...")
        !unzip -q {DATASET_ZIP} -d {DATASET_DIR}
        !rm {DATASET_ZIP}
    
    # Verify dataset structure
    required_files = [
        f"{DATASET_PATH}/transforms_train.json",
        f"{DATASET_PATH}/transforms_test.json",
        f"{DATASET_PATH}/train",
    ]
    
    for f in required_files:
        if not os.path.exists(f):
            raise FileNotFoundError(f"Missing: {f}")
    
    print(f"[OK] Dataset ready at {DATASET_PATH}")
    CHECKPOINT.mark_complete("section_b1_dataset")

In [ ]:
# Cell B.2: Display Sample Frames
import matplotlib.pyplot as plt
from PIL import Image

train_dir = Path(DATASET_PATH) / "train"
frames = sorted(train_dir.glob("*.png"))

print(f"Dataset contains {len(frames)} training frames")

# Show sample frames
if len(frames) >= 3:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    indices = [0, len(frames)//2, -1]
    for ax, idx in zip(axes, indices):
        img = Image.open(frames[idx])
        ax.imshow(img)
        ax.set_title(f"Frame {idx if idx >= 0 else len(frames)+idx}")
        ax.axis('off')
    plt.suptitle("D-NeRF Bouncingballs Dataset")
    plt.tight_layout()
    plt.show()

---
## Section C: Direct De3DGS Training

In [ ]:
# Cell C.1: Train De3DGS on Bouncingballs
import time

DE3DGS_DIR = f"{CONFIG.repo_dir}/deformable3dgs"
OUTPUT_DIR = f"{CONFIG.output_dir}/direct_training"

if CHECKPOINT.is_complete("section_c1_training"):
    print("[SKIP] Section C.1 already complete, skipping training...")
else:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    print(f"Starting De3DGS training...")
    print(f"  Dataset: {DATASET_PATH}")
    print(f"  Iterations: {CONFIG.training_iterations}")
    print(f"  Output: {OUTPUT_DIR}")
    print()
    
    start_time = time.time()
    
    # Run training
    !cd {DE3DGS_DIR} && python train.py \
        -s {DATASET_PATH} \
        -m {OUTPUT_DIR} \
        --eval \
        --is_blender \
        --iterations {CONFIG.training_iterations}
    
    elapsed = time.time() - start_time
    print(f"\nTraining completed in {elapsed/60:.1f} minutes")
    CHECKPOINT.mark_complete("section_c1_training")

In [ ]:
# Cell C.2: Validate Training Output
import json

# Check output structure
expected_outputs = [
    f"{OUTPUT_DIR}/point_cloud/iteration_{CONFIG.training_iterations}/point_cloud.ply",
    f"{OUTPUT_DIR}/deform/iteration_{CONFIG.training_iterations}/deform.pth",
]

print("=" * 50)
print("TRAINING OUTPUT VALIDATION")
print("=" * 50)

for path in expected_outputs:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"[OK] {os.path.basename(path)} ({size_mb:.1f} MB)")
    else:
        print(f"[FAIL] {path} not found!")

# Check for PSNR in training logs (if available)
# De3DGS outputs metrics to stdout during training
print("\n[INFO] Check training output above for PSNR values")
print(f"[THRESHOLD] PSNR should be > {CONFIG.min_psnr_synthetic} dB for bouncingballs")

---
## Section D: Adapter Integration Test

In [ ]:
# Cell D.1: Create Mock VideoProcessingResult
import sys
sys.path.insert(0, f"{CONFIG.repo_dir}/backend")

from stages.video_processing import VideoProcessingResult

# Mock result pointing to bouncingballs dataset
# For blender datasets, transforms are in a different format
mock_result = VideoProcessingResult(
    output_dir=DATASET_PATH,
    frames_dir=f"{DATASET_PATH}/train",
    transforms_path=f"{DATASET_PATH}/transforms_train.json",
    num_frames=len(list(Path(f"{DATASET_PATH}/train").glob("*.png"))),
    metadata={"is_blender": True},
)

print(f"Mock VideoProcessingResult:")
print(f"  output_dir: {mock_result.output_dir}")
print(f"  num_frames: {mock_result.num_frames}")

In [ ]:
# Cell D.2: Test Deformable3DGSAdapter
from adapters.deformable3dgs import Deformable3DGSAdapter, Deformable3DGSOptions

ADAPTER_OUTPUT = f"{CONFIG.output_dir}/adapter_test"

if CHECKPOINT.is_complete("section_d2_adapter"):
    print("[SKIP] Section D.2 already complete, skipping adapter test...")
else:
    options = Deformable3DGSOptions(
        iterations=CONFIG.training_iterations,
        is_blender=True,
        export_num_frames=30,
    )
    
    adapter = Deformable3DGSAdapter({})
    
    print("Running Deformable3DGSAdapter.run_full_pipeline()...")
    
    result = adapter.run_full_pipeline(
        video_result=mock_result,
        output_dir=ADAPTER_OUTPUT,
        options=options,
        progress_callback=lambda p, m: print(f"  [{p*100:5.1f}%] {m}")
    )
    
    print(f"\nAdapter result:")
    print(f"  PLY files: {len(result.ply_paths)}")
    print(f"  Model path: {result.model_path}")
    print(f"  Num Gaussians: {result.num_gaussians}")
    
    adapter.cleanup()
    CHECKPOINT.mark_complete("section_d2_adapter")

In [ ]:
# Cell D.3: Validate PLY Export Format
from plyfile import PlyData
import numpy as np

def validate_ply_format(ply_path: str) -> dict:
    """Validate a PLY file matches 3DGS specification."""
    errors = []
    warnings = []
    
    try:
        plydata = PlyData.read(ply_path)
    except Exception as e:
        return {"valid": False, "errors": [f"Failed to read PLY: {e}"]}
    
    vertex = plydata['vertex']
    num_gaussians = len(vertex.data)
    
    # Required properties
    required = ['x', 'y', 'z', 'opacity', 'scale_0', 'scale_1', 'scale_2',
                'rot_0', 'rot_1', 'rot_2', 'rot_3', 'f_dc_0', 'f_dc_1', 'f_dc_2']
    
    for prop in required:
        if prop not in vertex.data.dtype.names:
            errors.append(f"Missing required property: {prop}")
    
    # Check for NaN/Inf
    for prop in ['x', 'y', 'z', 'opacity']:
        if prop in vertex.data.dtype.names:
            data = vertex[prop]
            if not np.isfinite(data).all():
                errors.append(f"Property {prop} contains NaN/Inf values")
    
    # Check quaternion normalization
    if all(f'rot_{i}' in vertex.data.dtype.names for i in range(4)):
        quats = np.stack([vertex[f'rot_{i}'] for i in range(4)], axis=-1)
        norms = np.linalg.norm(quats, axis=-1)
        if not np.allclose(norms, 1.0, atol=0.1):
            warnings.append(f"Quaternions not normalized (mean norm: {norms.mean():.3f})")
    
    return {
        "valid": len(errors) == 0,
        "num_gaussians": num_gaussians,
        "errors": errors,
        "warnings": warnings,
    }

# Find PLY files
ply_dir = Path(ADAPTER_OUTPUT) / "per_frame_plys"
if ply_dir.exists():
    ply_files = sorted(ply_dir.glob("*.ply"))
else:
    # Fallback to direct training output
    ply_dir = Path(OUTPUT_DIR) / "point_cloud" / f"iteration_{CONFIG.training_iterations}"
    ply_files = list(ply_dir.glob("*.ply"))

print(f"Validating {len(ply_files)} PLY files...")

all_valid = True
for ply_path in ply_files[:5]:  # Sample first 5
    result = validate_ply_format(str(ply_path))
    status = "[OK]" if result["valid"] else "[FAIL]"
    print(f"  {status} {ply_path.name}: {result['num_gaussians']} Gaussians")
    if result["errors"]:
        print(f"      Errors: {result['errors']}")
    if result["warnings"]:
        print(f"      Warnings: {result['warnings']}")
    all_valid = all_valid and result["valid"]

if all_valid:
    print("\n[OK] All sampled PLY files valid")
    CHECKPOINT.mark_complete("section_d3_ply_validation")
else:
    print("\n[FAIL] PLY validation failed!")

---
## Section E: SE(3) 6-DoF Validation (Optional)

This section tests the `is_6dof=True` mode which uses SE(3) rigid transformations instead of additive deformations.

In [ ]:
# Cell E.1: SE(3) Comparison Test
# This is optional - skip if time-constrained

RUN_SE3_TEST = False  # Set to True to run SE(3) comparison

if not RUN_SE3_TEST:
    print("[SKIP] SE(3) test disabled. Set RUN_SE3_TEST = True to enable.")
else:
    print("Running SE(3) comparison test...")
    print("This trains an additional model with is_6dof=True")
    # Training code would go here
    pass

---
## Section F: Visualization & Reporting

In [ ]:
# Cell F.1: Visualize Gaussian Statistics
import matplotlib.pyplot as plt

if ply_files:
    counts = []
    for ply_path in ply_files:
        plydata = PlyData.read(str(ply_path))
        counts.append(len(plydata['vertex']))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Line plot
    axes[0].plot(counts, 'b-', linewidth=2)
    axes[0].fill_between(range(len(counts)), counts, alpha=0.3)
    axes[0].set_xlabel('Frame')
    axes[0].set_ylabel('Number of Gaussians')
    axes[0].set_title('Gaussians per Frame')
    axes[0].grid(True, alpha=0.3)
    
    # Histogram
    axes[1].hist(counts, bins=20, edgecolor='black', alpha=0.7)
    axes[1].axvline(np.mean(counts), color='r', linestyle='--', label=f'Mean: {np.mean(counts):,.0f}')
    axes[1].set_xlabel('Number of Gaussians')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Distribution of Gaussian Counts')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nGaussian count statistics:")
    print(f"  Min: {min(counts):,}")
    print(f"  Max: {max(counts):,}")
    print(f"  Mean: {np.mean(counts):,.0f}")
    print(f"  Std: {np.std(counts):,.0f}")
else:
    print("No PLY files found for visualization")

In [ ]:
# Cell F.2: Memory Usage Report
if torch.cuda.is_available():
    print("=" * 50)
    print("MEMORY USAGE REPORT")
    print("=" * 50)
    
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    max_allocated = torch.cuda.max_memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"Current allocated: {allocated:.2f} GB")
    print(f"Peak allocated: {max_allocated:.2f} GB")
    print(f"Reserved: {reserved:.2f} GB")
    print(f"Total VRAM: {total:.2f} GB")
    print(f"Peak usage: {max_allocated/total*100:.1f}%")
    
    if max_allocated > CONFIG.max_vram_gb:
        print(f"\n[WARN] Peak memory ({max_allocated:.1f} GB) exceeded threshold ({CONFIG.max_vram_gb} GB)")
    else:
        print(f"\n[OK] Memory usage within limits")

---
## Section G: User Video Test (Optional)

Upload your own video to test the full Stage 1 + Stage 3 pipeline.

In [ ]:
# Cell G.1: Upload Video from Local Drive
"""
Upload your own video to test the full pipeline.
Supported formats: MP4, MOV, AVI, WEBM
Recommended: MP4 with H.264 codec, 720p-1080p, 2-10 seconds
"""
from google.colab import files
import subprocess

RUN_USER_VIDEO_TEST = False  # Set to True to enable user video testing

USER_VIDEO_DIR = f"{CONFIG.output_dir}/user_video"
USER_VIDEO_PATH = None  # Will be set after upload

if not RUN_USER_VIDEO_TEST:
    print("[SKIP] User video test disabled.")
    print("Set RUN_USER_VIDEO_TEST = True and re-run to enable.")
else:
    print("=" * 50)
    print("VIDEO UPLOAD")
    print("=" * 50)
    print("\nSupported formats: MP4, MOV, AVI, WEBM")
    print("Recommended: MP4 with H.264 codec, 720p-1080p, 2-10 seconds\n")
    
    uploaded = files.upload()
    
    if uploaded:
        os.makedirs(USER_VIDEO_DIR, exist_ok=True)
        
        for filename, data in uploaded.items():
            video_path = f"{USER_VIDEO_DIR}/{filename}"
            with open(video_path, 'wb') as f:
                f.write(data)
            
            # Get video info using ffprobe
            result = subprocess.run([
                'ffprobe', '-v', 'quiet', '-print_format', 'json',
                '-show_format', '-show_streams', video_path
            ], capture_output=True, text=True)
            
            if result.returncode == 0:
                import json as json_lib
                info = json_lib.loads(result.stdout)
                duration = float(info['format'].get('duration', 0))
                video_stream = next((s for s in info['streams'] if s['codec_type'] == 'video'), {})
                width = video_stream.get('width', 'unknown')
                height = video_stream.get('height', 'unknown')
                fps = eval(video_stream.get('r_frame_rate', '30/1'))
                
                print(f"\n[OK] Video uploaded: {filename}")
                print(f"     Resolution: {width}x{height}")
                print(f"     Duration: {duration:.1f} seconds")
                print(f"     Frame rate: {fps:.1f} fps")
                print(f"     Estimated frames: {int(duration * fps)}")
                
                USER_VIDEO_PATH = video_path
            else:
                print(f"[WARN] Could not read video metadata for {filename}")
                USER_VIDEO_PATH = video_path
                
            # Display first frame preview
            from IPython.display import display, Image as IPImage
            preview_path = f"{USER_VIDEO_DIR}/preview.jpg"
            subprocess.run([
                'ffmpeg', '-y', '-i', video_path, '-vframes', '1',
                '-q:v', '2', preview_path
            ], capture_output=True)
            
            if os.path.exists(preview_path):
                print("\nFirst frame preview:")
                display(IPImage(filename=preview_path, width=400))
    else:
        print("[SKIP] No video uploaded.")
        USER_VIDEO_PATH = None

---
## Section H: Download Results

Download your training results including PLY files, rendered MP4 videos, and validation reports.

In [ ]:
# Cell H.1: Render 4D Gaussian Visualization MP4
"""
Render the trained De3DGS model into an MP4 video.
This creates a visualization showing the 4D Gaussians over time.
"""
from pathlib import Path

RENDER_OUTPUT_DIR = f"{CONFIG.output_dir}/renders"
os.makedirs(RENDER_OUTPUT_DIR, exist_ok=True)

# Find the training output directory
training_output = Path(CONFIG.output_dir) / "direct_training"

if not training_output.exists():
    print("[SKIP] No training output found. Run Section C first.")
else:
    print("=" * 50)
    print("RENDERING 4D VISUALIZATION")
    print("=" * 50)
    
    # Use De3DGS render script
    render_script = f"{CONFIG.repo_dir}/deformable3dgs/render.py"
    
    if os.path.exists(render_script):
        print("\nRunning De3DGS renderer...")
        print("This may take a few minutes...\n")
        
        !cd {CONFIG.repo_dir}/deformable3dgs && python render.py \
            -m {training_output} \
            --skip_train
        
        # Find rendered frames
        render_dir = training_output / "test" / f"ours_{CONFIG.training_iterations}"
        if render_dir.exists():
            render_frames = sorted(render_dir.glob("*.png"))
            
            if render_frames:
                print(f"\n[OK] Rendered {len(render_frames)} frames")
                
                # Create MP4 from rendered frames
                mp4_path = f"{RENDER_OUTPUT_DIR}/4d_gaussian_render.mp4"
                
                !ffmpeg -y -framerate 15 \
                    -pattern_type glob -i '{render_dir}/*.png' \
                    -c:v libx264 -pix_fmt yuv420p \
                    -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" \
                    {mp4_path} 2>/dev/null
                
                if os.path.exists(mp4_path):
                    mp4_size = os.path.getsize(mp4_path) / 1e6
                    print(f"[OK] 4D visualization MP4 created: {mp4_path}")
                    print(f"     Size: {mp4_size:.1f} MB")
                    print(f"     Frames: {len(render_frames)}")
                    
                    # Display in notebook
                    from IPython.display import Video, display
                    print("\nPlaying rendered video:")
                    display(Video(mp4_path, embed=True, width=640))
                else:
                    print("[WARN] MP4 creation failed - ffmpeg may need adjustment")
            else:
                print("[WARN] No rendered frames found")
        else:
            print(f"[WARN] Render directory not found: {render_dir}")
    else:
        print("[INFO] De3DGS render script not found")
        print("       Skipping MP4 rendering")

In [ ]:
# Cell H.2: Package All Results into Downloadable ZIP
"""
Package PLY files, MP4 videos, model checkpoints, and reports
into a single ZIP file for download.
"""
import zipfile
from pathlib import Path

DOWNLOAD_ZIP = "/content/brain_dance_results.zip"

print("=" * 50)
print("PACKAGING RESULTS")
print("=" * 50)

files_added = 0

with zipfile.ZipFile(DOWNLOAD_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    
    # Add PLY files
    print("\nAdding PLY files...")
    for ply_file in Path(CONFIG.output_dir).rglob("*.ply"):
        arcname = f"ply_files/{ply_file.parent.name}/{ply_file.name}"
        zf.write(ply_file, arcname)
        files_added += 1
        if files_added <= 5:  # Only show first 5
            print(f"  + {arcname}")
    if files_added > 5:
        print(f"  ... and {files_added - 5} more PLY files")
    
    # Add MP4 renders
    print("\nAdding MP4 videos...")
    mp4_count = 0
    for mp4_file in Path(CONFIG.output_dir).rglob("*.mp4"):
        arcname = f"videos/{mp4_file.name}"
        zf.write(mp4_file, arcname)
        files_added += 1
        mp4_count += 1
        print(f"  + {arcname} ({os.path.getsize(mp4_file) / 1e6:.1f} MB)")
    
    if mp4_count == 0:
        print("  (No MP4 files found - run H.1 first)")
    
    # Add model checkpoints
    print("\nAdding model checkpoints...")
    pth_count = 0
    for pth_file in Path(CONFIG.output_dir).rglob("*.pth"):
        arcname = f"models/{pth_file.parent.name}/{pth_file.name}"
        zf.write(pth_file, arcname)
        files_added += 1
        pth_count += 1
        print(f"  + {arcname} ({os.path.getsize(pth_file) / 1e6:.1f} MB)")
    
    if pth_count == 0:
        print("  (No .pth files found)")
    
    # Add validation report
    print("\nAdding validation report...")
    report_path = f"{CONFIG.output_dir}/validation_report.json"
    if os.path.exists(report_path):
        zf.write(report_path, "validation_report.json")
        files_added += 1
        print("  + validation_report.json")
    else:
        # Create one now
        report = {
            "timestamp": datetime.now().isoformat(),
            "gpu": gpu_info if 'gpu_info' in dir() else {},
            "config": {
                "training_iterations": CONFIG.training_iterations,
                "branch": CONFIG.branch,
            },
            "completed_sections": CHECKPOINT.state["completed_sections"],
        }
        zf.writestr("validation_report.json", json.dumps(report, indent=2))
        files_added += 1
        print("  + validation_report.json (generated)")

# Report final size
zip_size_mb = os.path.getsize(DOWNLOAD_ZIP) / 1e6
print(f"\n" + "=" * 50)
print(f"[OK] Created: {DOWNLOAD_ZIP}")
print(f"     Total files: {files_added}")
print(f"     Size: {zip_size_mb:.1f} MB")
print("=" * 50)

In [ ]:
# Cell H.3: Download Results to Local Machine
"""
Download the packaged ZIP file to your local machine.
Your browser will prompt you to save the file.
"""
from google.colab import files

print("=" * 50)
print("DOWNLOAD RESULTS")
print("=" * 50)
print()

if os.path.exists(DOWNLOAD_ZIP):
    zip_size = os.path.getsize(DOWNLOAD_ZIP) / 1e6
    print(f"Downloading: brain_dance_results.zip")
    print(f"Size: {zip_size:.1f} MB")
    print()
    print("Your browser will prompt you to save the file...")
    print()
    
    files.download(DOWNLOAD_ZIP)
    
    print("\n[OK] Download initiated!")
    print()
    print("ZIP contents:")
    print("  ply_files/          - Per-frame PLY Gaussian files")
    print("  videos/             - 4D visualization MP4s")
    print("  models/             - Trained model checkpoints (.pth)")
    print("  validation_report.json - Test results summary")
    print()
    print("To view the 4D Gaussians:")
    print("  1. Use a PLY viewer like MeshLab or CloudCompare")
    print("  2. Or convert to .splat format for web viewers")
else:
    print("[FAIL] No results ZIP found.")
    print("       Run Cell H.2 first to package the results.")

---
## Section I: Cleanup & Summary

In [ ]:
# Cell I.1: Generate Validation Report
"""
Generate a comprehensive validation report with all test results.
"""
import json
from datetime import datetime

report = {
    "timestamp": datetime.now().isoformat(),
    "notebook_version": "1.1.0",
    "gpu": gpu_info if 'gpu_info' in dir() else {},
    "config": {
        "training_iterations": CONFIG.training_iterations,
        "branch": CONFIG.branch,
        "min_psnr_threshold": CONFIG.min_psnr_synthetic,
    },
    "completed_sections": CHECKPOINT.state["completed_sections"],
    "results": {
        "ply_files_validated": len(ply_files) if 'ply_files' in dir() else 0,
        "all_valid": all_valid if 'all_valid' in dir() else None,
    },
    "artifacts": {
        "zip_path": DOWNLOAD_ZIP if os.path.exists(DOWNLOAD_ZIP) else None,
        "zip_size_mb": os.path.getsize(DOWNLOAD_ZIP) / 1e6 if os.path.exists(DOWNLOAD_ZIP) else 0,
    }
}

report_path = f"{CONFIG.output_dir}/validation_report.json"
os.makedirs(CONFIG.output_dir, exist_ok=True)
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"Validation report saved to: {report_path}")
print()
print("Report summary:")
print(f"  Timestamp: {report['timestamp']}")
print(f"  Completed sections: {len(report['completed_sections'])}")
print(f"  PLY files validated: {report['results']['ply_files_validated']}")

In [ ]:
# Cell I.2: Final Summary
"""
Display final validation summary with pass/fail status for all gates.
"""
print("=" * 60)
print("DE3DGS VALIDATION SUMMARY")
print("=" * 60)
print()

# Gate 1: Environment
gate1 = CHECKPOINT.is_complete("section_a5_verify")
print(f"Gate 1 (Environment):    {'PASS' if gate1 else 'FAIL'}")

# Gate 2: Training
gate2 = CHECKPOINT.is_complete("section_c1_training")
print(f"Gate 2 (Training):       {'PASS' if gate2 else 'FAIL'}")

# Gate 3: PLY Export
gate3 = CHECKPOINT.is_complete("section_d3_ply_validation")
print(f"Gate 3 (PLY Export):     {'PASS' if gate3 else 'FAIL'}")

# Gate 4: Adapter (optional)
gate4 = CHECKPOINT.is_complete("section_d2_adapter")
print(f"Gate 4 (Adapter):        {'PASS' if gate4 else 'SKIP'}")

# Overall
all_gates_pass = gate1 and gate2 and gate3
print()
print("=" * 60)
if all_gates_pass:
    print("VALIDATION COMPLETE - ALL REQUIRED GATES PASSED")
else:
    print("VALIDATION INCOMPLETE - CHECK FAILED GATES ABOVE")
print("=" * 60)
print()
print(f"Completed sections: {CHECKPOINT.state['completed_sections']}")
print()

if os.path.exists(DOWNLOAD_ZIP):
    print(f"Results available: {DOWNLOAD_ZIP}")
    print("Run Cell H.3 to download to your local machine.")
else:
    print("Run Section H to package and download results.")